<a href="https://colab.research.google.com/github/silvatria/pembelajaran-mesin-254107023001/blob/main/JS09_TL02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu annoy hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 21.9 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520957 sha256=1e7f3cd940f801346d5f4e6f808c4273642d22d0cc94d986b062af184459c091
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679067 sha256=a737532b5e18574a6b405966e195a807fec5999614488a5fda68c42777f21743
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [9]:
import os
import pandas as pd
import numpy as np
import time
import kagglehub
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# 1. Download & Load Dataset Otomatis
# -------------------------------
print("Sedang mendownload dataset dari Kaggle...")
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

# Pilih file dataset pertama (yang memiliki atribut lagu)
files = os.listdir(path)
target_file = [f for f in files if 'attributes' in f][0]
csv_path = os.path.join(path, target_file)

print(f"Memuat file CSV: {target_file}...")
df = pd.read_csv(csv_path)

# Debug: Print kolom yang tersedia di DataFrame
print("Kolom tersedia di dataset:", df.columns.tolist())

# -------------------------------
# 2. Preprocessing & Feature Scaling
# -------------------------------
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Bersihkan baris yang mengandung missing value pada fitur terpilih
df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values.astype('float32')

# Standarisasi fitur menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

n_samples, d = X_scaled.shape
k = 10  # jumlah nearest neighbors

print(f"Total data bersih: {n_samples:,} baris, {d} dimensi fitur.")

# Ambil 1 lagu acak/pertama sebagai query pencarian
query_idx = 0
query_vector = X_scaled[query_idx:query_idx+1]

# Deteksi nama kolom judul lagu secara dinamis
possible_name_cols = ['track_name', 'name', 'song_name', 'title']
col_name = next((col for col in possible_name_cols if col in df.columns), df.columns[0])
query_song_name = df.iloc[query_idx].get(col_name, f'Index {query_idx}')

print(f"Query Track: '{query_song_name}' (Index: {query_idx})\n")


# -------------------------------
# 3. Exact Nearest Neighbors (Baseline)
# -------------------------------
t0 = time.time()
nn_exact = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn_exact.fit(X_scaled)
t_index_exact = time.time() - t0

t0 = time.time()
distances_exact, indices_exact = nn_exact.kneighbors(query_vector)
t_search_exact = time.time() - t0
ground_truth = indices_exact[0]


# -------------------------------
# 4. ANNOY
# -------------------------------
t0 = time.time()
annoy_index = AnnoyIndex(d, 'euclidean')
for i, vec in enumerate(X_scaled):
    annoy_index.add_item(i, vec)
annoy_index.build(n_trees=20)  # 20 trees trade-off kecepatan & akurasi
t_index_annoy = time.time() - t0

t0 = time.time()
indices_annoy = annoy_index.get_nns_by_vector(query_vector[0], k)
t_search_annoy = time.time() - t0


# -------------------------------
# 5. FAISS (IndexFlatL2)
# -------------------------------
t0 = time.time()
faiss_index = faiss.IndexFlatL2(d)
faiss_index.add(X_scaled)
t_index_faiss = time.time() - t0

t0 = time.time()
distances_faiss, indices_faiss = faiss_index.search(query_vector, k)
t_search_faiss = time.time() - t0
indices_faiss = indices_faiss[0]


# -------------------------------
# 6. HNSWLIB
# -------------------------------
t0 = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=d)
hnsw_index.init_index(max_elements=n_samples, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled, np.arange(n_samples))
hnsw_index.set_ef(50)
t_index_hnsw = time.time() - t0

t0 = time.time()
indices_hnsw, distances_hnsw = hnsw_index.knn_query(query_vector, k=k)
t_search_hnsw = time.time() - t0
indices_hnsw = indices_hnsw[0]


# -------------------------------
# 7. Evaluasi & Tabel Perbandingan
# -------------------------------
def calc_overlap(pred, target):
    return len(set(pred).intersection(set(target))) / len(target) * 100

summary_data = [
    {
        "Algoritma": "Exact NN (Baseline)",
        "Build Time (s)": f"{t_index_exact:.4f}",
        "Search Time (s)": f"{t_search_exact:.6f}",
        "Akurasi / Overlap (%)": "100.00%"
    },
    {
        "Algoritma": "ANNOY",
        "Build Time (s)": f"{t_index_annoy:.4f}",
        "Search Time (s)": f"{t_search_annoy:.6f}",
        "Akurasi / Overlap (%)": f"{calc_overlap(indices_annoy, ground_truth):.2f}%"
    },
    {
        "Algoritma": "FAISS (FlatL2)",
        "Build Time (s)": f"{t_index_faiss:.4f}",
        "Search Time (s)": f"{t_search_faiss:.6f}",
        "Akurasi / Overlap (%)": f"{calc_overlap(indices_faiss, ground_truth):.2f}%"
    },
    {
        "Algoritma": "HNSWLIB",
        "Build Time (s)": f"{t_index_hnsw:.4f}",
        "Search Time (s)": f"{t_search_hnsw:.6f}",
        "Akurasi / Overlap (%)": f"{calc_overlap(indices_hnsw, ground_truth):.2f}%"
    }
]

df_summary = pd.DataFrame(summary_data)
print("\n=== TABEL PERBANDINGAN PERFORMA ALGORITMA ===")
print(df_summary.to_markdown(index=False))

# Tampilkan hasil rekomendasi lagu terdekat dengan deteksi kolom otomatis
print("\n=== TOP-10 TRACK REKOMENDASI (Exact vs ANN) ===")
def print_tracks(name, idxs):
    print(f"\n--- {name} ---")
    for r, i in enumerate(idxs, 1):
        row = df.iloc[i]

        # Cari kolom lagu dan artis secara aman
        t_name = 'N/A'
        for col in ['track_name', 'name', 'song_name', 'title']:
            if col in df.columns:
                t_name = row.get(col, 'N/A')
                break

        t_artist = 'N/A'
        for col in ['track_artist', 'artist', 'artist_name', 'singer']:
            if col in df.columns:
                t_artist = row.get(col, 'N/A')
                break

        print(f"{r:2d}. {t_name} - {t_artist}")

print_tracks("Exact NN (Ground Truth)", ground_truth)
print_tracks("ANNOY", indices_annoy)
print_tracks("FAISS", indices_faiss)
print_tracks("HNSWLIB", indices_hnsw)

# === TABEL PERBANDINGAN PERFORMA ALGORITMA ===
# | Algoritma           |   Build Time (s) |   Search Time (s) | Akurasi / Overlap (%)   |
# |:--------------------|-----------------:|------------------:|:------------------------|
# | Exact NN (Baseline) |           0.0135 |          0.029314 | 100.00%                 |
# | ANNOY               |          19.1298 |          0.000282 | 100.00%                 |
# | FAISS (FlatL2)      |           0.0193 |          0.009017 | 100.00%                 |
# | HNSWLIB             |         147.275  |          0.000237 | 100.00%                 |

# === TOP-10 TRACK REKOMENDASI (Exact vs ANN) ===

# --- Exact NN (Ground Truth) ---
#  1. ! - N/A
#  2. Intro (Who The Fuck Are You') - N/A
#  3. Testimonium - N/A
#  4. Ultimate Satisfaction - The Field Remix - N/A
#  5. Sæglópur - N/A
#  6. Autumn Interlude - N/A
#  7. Púrpura - N/A
#  8. Turn from Heaven - N/A
#  9. See You When I Die - N/A
# 10. Mrityu - N/A

# --- ANNOY ---
#  1. ! - N/A
#  2. Intro (Who The Fuck Are You') - N/A
#  3. Testimonium - N/A
#  4. Ultimate Satisfaction - The Field Remix - N/A
#  5. Sæglópur - N/A
#  6. Autumn Interlude - N/A
#  7. Púrpura - N/A
#  8. Turn from Heaven - N/A
#  9. See You When I Die - N/A
# 10. Mrityu - N/A

# --- FAISS ---
#  1. ! - N/A
#  2. Intro (Who The Fuck Are You') - N/A
#  3. Testimonium - N/A
#  4. Ultimate Satisfaction - The Field Remix - N/A
#  5. Sæglópur - N/A
#  6. Autumn Interlude - N/A
#  7. Púrpura - N/A
#  8. Turn from Heaven - N/A
#  9. See You When I Die - N/A
# 10. Mrityu - N/A

# --- HNSWLIB ---
#  1. ! - N/A
#  2. Intro (Who The Fuck Are You') - N/A
#  3. Testimonium - N/A
#  4. Ultimate Satisfaction - The Field Remix - N/A
#  5. Sæglópur - N/A
#  6. Autumn Interlude - N/A
#  7. Púrpura - N/A
#  8. Turn from Heaven - N/A
#  9. See You When I Die - N/A
# 10. Mrityu - N/A

Sedang mendownload dataset dari Kaggle...
Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Memuat file CSV: songs_with_attributes_and_lyrics.csv...
Kolom tersedia di dataset: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']
Total data bersih: 955,320 baris, 9 dimensi fitur.
Query Track: '!' (Index: 0)


=== TABEL PERBANDINGAN PERFORMA ALGORITMA ===
| Algoritma           |   Build Time (s) |   Search Time (s) | Akurasi / Overlap (%)   |
|:--------------------|-----------------:|------------------:|:------------------------|
| Exact NN (Baseline) |           0.0135 |          0.029314 | 100.00%                 |
| ANNOY               |          19.1298 |          0.000282 | 100.00%                 |
| FAISS (FlatL2)      |           0.0193 |          0.009017 | 100.00%                 |
| HNSW